---

###  Dataset Overview

The dataset consists of **mobile phone specifications** with the goal of predicting the **`price_range`**, which categorizes phones into four price classes:

* **0**: Low cost
* **1**: Medium cost
* **2**: High cost
* **3**: Very high cost

Each row represents one mobile phone, and the dataset is structured with **numerical features** that describe various technical attributes of the device.

---

###  Feature Summary

| Feature Name           | Description                                               |
| ---------------------- | --------------------------------------------------------- |
| `battery_power`        | Battery capacity in mAh                                   |
| `blue`                 | Whether the phone has Bluetooth (1 = Yes, 0 = No)         |
| `clock_speed`          | Clock speed of the processor in GHz                       |
| `dual_sim`             | Whether the phone has dual SIM support (1 = Yes, 0 = No)  |
| `fc`                   | Front camera megapixels                                   |
| `four_g`               | Whether the phone supports 4G (1 = Yes, 0 = No)           |
| `int_memory`           | Internal memory in GB                                     |
| `m_dep`                | Mobile depth in cm                                        |
| `mobile_wt`            | Weight of the mobile phone in grams                       |
| `n_cores`              | Number of processor cores                                 |
| `pc`                   | Primary camera megapixels                                 |
| `px_height`            | Height resolution of the screen in pixels                 |
| `px_width`             | Width resolution of the screen in pixels                  |
| `ram`                  | Random Access Memory (RAM) in MB                          |
| `sc_h`                 | Screen height in cm                                       |
| `sc_w`                 | Screen width in cm                                        |
| `talk_time`            | Maximum talk time on battery in hours                     |
| `three_g`              | Whether the phone supports 3G (1 = Yes, 0 = No)           |
| `touch_screen`         | Whether the phone has a touchscreen (1 = Yes, 0 = No)     |
| `wifi`                 | Whether the phone supports WiFi (1 = Yes, 0 = No)         |
| `price_range` (target) | Price category (0: Low, 1: Medium, 2: High, 3: Very High) |

---

# Step 1: Import libraries

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
import warnings
warnings.filterwarnings("ignore")
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Step 2: Load dataset

In [ ]:
df = pd.read_csv("/kaggle/input/mobile-phone-pricing-dataset/Mobile Phone Pricing.csv")

# Step 3: Initial data check

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.nunique()

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

# Step 4: Remove duplicates and check again

In [ ]:
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

# Step 5: Data Summary

In [ ]:
df.describe()

# Step 6: Feature summary

In [ ]:
df.columns.tolist()

# Step 7: Correlation heatmap

In [ ]:
plt.figure(figsize=(20, 16))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

# Step 8: Pairplot for numeric insights

In [ ]:
sns.pairplot(df.select_dtypes(include=['int64', 'float64']))
plt.suptitle("Feature Distributions and Relations", y=1.02)
plt.show()


# Step 9: Check price distribution

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df['price_range'], kde=True, bins=30)
plt.title("Distribution of Mobile Phone Prices")
plt.xlabel("price_range")
plt.ylabel("Count")
plt.show()

# Step 10: Define features and target

In [ ]:
X = df.drop("price_range", axis=1)
y = df["price_range"]

# Encode categorical columns if any

In [ ]:
X = pd.get_dummies(X, drop_first=True)

# Step 11: Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=537)


# Step 12: Scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 13: Define models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=537),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=537),
    "XGBoost": XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metric='mlogloss'),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "SVC (RBF Kernel)": SVC(kernel='rbf', probability=True)
}


# Step 14: Train and evaluate each model

In [ ]:
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"\n{name} Accuracy: {acc:.4f}")
    print("Classification Report:\n", classification_report(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap="Blues")
    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

# Step 15: Model Comparison Visualization

In [ ]:
results_df = pd.DataFrame.from_dict(results, orient='index', columns=['Accuracy'])
results_df = results_df.sort_values(by="Accuracy", ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=results_df.index, y="Accuracy", data=results_df)
plt.title("Model Accuracy Comparison")
plt.xticks(rotation=45)
plt.ylabel("Accuracy")
plt.ylim(0.8, 1.0)
plt.show()

---

### **Project Summary**

**Step 1 – Importing Libraries**
Essential Python libraries were imported:

* `pandas` and `numpy` for data handling,
* `matplotlib` and `seaborn` for visualization,
* `scikit-learn` for preprocessing, model building, and evaluation,
* `xgboost` for advanced gradient boosting classification.

---

**Step 2 – Loading the Dataset**
The dataset `Mobile Phone Pricing.csv` was loaded using `pandas.read_csv()`.
Initial shape and structure were checked to ensure successful data loading.

---

**Step 3 – Initial Inspection**
Missing values and data types were printed to assess data quality.
The dataset was found to be clean and ready for analysis.

---

**Step 4 – Removing Duplicates**
Duplicate rows were removed to prevent data redundancy and improve model generalization.

---

**Step 5 – Statistical Summary**
Descriptive statistics (mean, std, min, max) were printed for all numeric features.
This provided insights into feature ranges and variability.

---

**Step 6 – Feature List Review**
All column names were listed.
The target variable `price_range` represents pricing levels:

* 0: Low,
* 1: Medium,
* 2: High,
* 3: Very High.

---

**Step 7 – Correlation Heatmap**
A correlation heatmap was plotted to explore relationships between features.
This helped identify potentially redundant or impactful variables.

---

**Step 8 – Pairplot Analysis**
Pairplots were used to visualize distributions and feature interactions.
This enabled visual assessment of class separability with respect to `price_range`.

---

**Step 9 – Target Variable Distribution**
A histogram of the `price_range` variable was plotted to verify class balance.
The classes appeared reasonably balanced across all four price categories.

---

**Step 10 – Feature/Target Definition**
Independent variables (`X`) and the dependent variable (`y`) were defined.
Categorical values (if any) were handled using one-hot encoding.

---

**Step 11 – Train-Test Split**
The dataset was split into training and test sets (80/20) using `train_test_split()`, with randomization for reproducibility.

---

**Step 12 – Feature Scaling**
StandardScaler was used to standardize numerical features.
This ensured uniform feature scaling to support certain algorithms (e.g., KNN, SVC).

---

**Step 13 – Model Definition**
Six widely used classification models were defined:

* Logistic Regression
* Random Forest Classifier
* Gradient Boosting Classifier
* XGBoost Classifier
* K-Nearest Neighbors
* Support Vector Classifier (RBF Kernel)

---

**Step 14 – Model Training and Evaluation**
Each model was trained and evaluated using the test set.
Key performance metrics included:

* **Accuracy Score**
* **Classification Report** (Precision, Recall, F1-Score)
* **Confusion Matrix**

---

**Step 15 – Model Comparison Visualization**
Model accuracies were stored and visualized using a barplot.
This allowed for easy comparison, where **XGBoost** delivered the highest classification accuracy.

---

# Thank you for taking the time to review my work. I would be very happy if you could upvote! 😊

---
